In [ ]:
import glob
import re
import numpy as np

from matplotlib import pyplot as plt

## Display Case 2 Results

This notebook generates Figures 4, 5, 6, and 7 for Case 2 (three-block model).

Before running this notebook, please execute the following script:

```
./run.sh
```


In [ ]:
plt.rcParams['font.size'] = 14
mmin = 0
mmax = 2.5
cwmin = -0.1
cwmax = 0.3

In [ ]:
def cross_section_index(direction, cross, x, y, z):
    '''
    Return indices of grid points corresponding to a specified cross-section.

    Parameters
    ----------
    direction : {'ew', 'ns', 'h'}
        Direction of the cross-section:
        'ew' : x–z section (fixed y),
        'ns' : y–z section (fixed x),
        'h'  : horizontal slice (fixed z).
    cross : float
        Target coordinate value. The nearest grid value is used.
    x, y, z : array_like
        Coordinates of the model grid.

    Returns
    -------
    idx : ndarray
        Indices of points belonging to the selected cross-section.

    Notes
    -----
    The cross-section is extracted at the grid location closest to
    the specified coordinate to ensure robustness against floating-point
    mismatch and non-uniform grids.
    '''
    direction = direction.lower()

    x0 = np.unique(x)
    y0 = np.unique(y)
    z0 = np.unique(z)

    def grid_spacing(arr, name):
        if len(arr) < 2:
            raise ValueError(f"{name} grid has insufficient points")
        return np.min(np.diff(arr))

    if direction == 'ew':
        # x(EW)-z cross section
        dy = grid_spacing(y0, "y")
        idx = np.where(np.abs(y - cross) < dy / 2.0)[0]

    elif direction == 'ns':
        # y(NS)-z cross section
        dx = grid_spacing(x0, "x")
        idx = np.where(np.abs(x - cross) < dx / 2.0)[0]

    elif direction == 'h':
        # horizontal slice
        dz = grid_spacing(z0, "z")
        idx = np.where(np.abs(z - cross) < dz / 2.0)[0]

    else:
        raise ValueError("direction must be one of {'ew', 'ns', 'h'}")

    if idx.size == 0:
        raise ValueError("No points found within the specified cross-section range")

    return idx

### True model

In [ ]:
res = np.loadtxt('../true.model', delimiter='\t')

x = res[:, 0]
y = res[:, 1]
z = res[:, 2]
b0 = res[:, 3]

x_unique = np.unique(x)
y_unique = np.unique(y)
z_unique = np.unique(z)

nx = len(x_unique)
ny = len(y_unique)
nz = len(z_unique)

print(nx, ny, nz)

In [ ]:
### Figure 8 (a) - (f) ###

# --- plot ---
fig = plt.figure(figsize=(12, 12))

for i in range(6):
    res = np.loadtxt('model_AdaL1TV_{:03d}.data'.format(i), delimiter='\t')
    x = res[:, 0]
    y = res[:, 1]
    z = res[:, 2]
    b = res[:, 3]

    yc0 = 0.0
    yc = y_unique[np.argmin(np.abs(y_unique - yc0))]
    idx1 = cross_section_index('EW', yc, x, y, z)

    # sort（z -> x）
    idx_sort = np.lexsort((x[idx1], z[idx1]))

    xs = x[idx1][idx_sort]
    zs = z[idx1][idx_sort]
    bs = b[idx1][idx_sort]

    xx = xs.reshape(nz, nx)
    zz = zs.reshape(nz, nx)
    bb = bs.reshape(nz, nx)

    # EW
    ax = fig.add_subplot(3, 2, i + 1)
    pc = ax.pcolormesh(xx, -zz, bb, cmap='jet', shading='auto')
    pc.set_clim(mmin, mmax)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(2.5, 0)
    ax.set_aspect('equal')
    ax.set_xlabel('y (km)')
    ax.set_ylabel('z (km)')
    plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.25, label='(A/m)')

plt.show()